# Introduction

// TODO

## Dataset

[CommonsenseQA](https://huggingface.co/datasets/tau/commonsense_qa) is a multiple-choice question answering dataset that requires common sense knowledge to select the correct answer from five choices.

# Setup

## Imports

This section contains all the imports that are required in this notebook and gives a brief overview which packages are being used.

In [1]:
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from huggingface_hub import hf_hub_download
from datasets import load_dataset

import nltk
from nltk.tokenize import word_tokenize
import gensim
import gensim.downloader as api

import wandb

## Seed

Setting a fixed random seed ensures **reproducibility** of results across runs. We set the seed for NumPy and PyTorch random number generators to ensure that random operations like weight initialization and data shuffling produce the same results each time the notebook is executed.

In [5]:
SEED = 5

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.mps.manual_seed(SEED)

## Experiment Tracking

[Weights and Biases](https://wandb.ai/site/) is a machine learning experiment tracking and visualization platform that helps track, compare, and optimize models.

The dashboard is available here: // TODO: add link to view for project 2

In [3]:
wandb.login()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: timon-schmid (timon-schmid-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

The `initialize_wandb_run` function configures a new wandb run with metadata about the experiment.

In [ ]:
# TODO: add missing parameters that are needed for tracking transformer models
def initialize_wandb_run(
    run_name,
    learning_rate,
    batch_size,
    num_epochs,
    entity_name="timon-schmid-hochschule-luzern",
    project_name="hslu-nlp-commonsense_qa",):
    
    wandb.init(
        entity=entity_name,
        project=project_name,
        name=run_name,
        config={
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "num_epochs": num_epochs,
            
            # Additional configuration details as per project requirements
            "dataset": "tau/commonsense_qa",
        }
    )
    
    # Return the wandb run object for further logging
    return wandb

# Preprocessing

## Tokenization

In [6]:
# TODO: 

## Data Loading

### Data Split

We use the data split that was mention in the course lecture, because the test set does not contain the answer key.

In [9]:
train_dataset = load_dataset("tau/commonsense_qa", split="train[:-1000]")
validation_dataset = load_dataset("tau/commonsense_qa", split="train[-1000:]")
test_dataset = load_dataset("tau/commonsense_qa", split="validation")

print(f'Train: {len(train_dataset)}, Validation: {len(validation_dataset)}, Test: {len(test_dataset)}')

Train: 8741, Validation: 1000, Test: 1221


# Model

## Randomly Initialized Transformer

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, d_model=768, n_head=12, num_layers=6, dim_feedforward=3072, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_head,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
    def forward(self, x, mask=None):
        return self.transformer_encoder(x, src_key_padding_mask=mask)

class FastTextEmbeddingLayer(nn.Module):
    def __init__(self, tokenizer, d_model=300, max_position_embeddings=512):
        super().__init__()
        
        # Access the pretrained FastText model from the tokenizer
        self.fasttext_model = tokenizer.fasttext_model
        
        # Initialize embedding matrix with FastText vectors
        self.d_model = d_model
        embedding_matrix = torch.zeros((tokenizer.vocab_size, d_model))
        
        # Set special token embeddings to random values
        for token in tokenizer.special_tokens:
            embedding_matrix[tokenizer.vocab[token]] = torch.randn(d_model) * 0.02
        
        # Fill embedding matrix with FastText vectors for words in vocabulary
        for word, idx in tokenizer.vocab.items():
            if word in self.fasttext_model and word not in tokenizer.special_tokens:
                embedding_matrix[idx] = torch.tensor(self.fasttext_model[word])
        
        # Create embeddings from pretrained weights and freeze them
        self.word_embeddings = nn.Embedding.from_pretrained(embedding_matrix, freeze=True)
        
        # Position embeddings are still trainable
        self.position_embeddings = nn.Embedding(max_position_embeddings, d_model)
        
        self.LayerNorm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)
        
    def forward(self, input_ids):
        seq_length = input_ids.size(1)
        position_ids = torch.arange(seq_length, dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)
        
        word_embeddings = self.word_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        
        embeddings = word_embeddings + position_embeddings
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        
        return embeddings

class RandomTransformer(nn.Module):
    def __init__(self, tokenizer, d_model=300, n_head=6, num_layers=6, 
                 dim_feedforward=1200, num_choices=5, dropout=0.1):
        super().__init__()
        self.num_choices = num_choices
        self.d_model = d_model
        
        # Embedding layer with pretrained FastText
        self.embeddings = FastTextEmbeddingLayer(tokenizer, d_model=d_model)
        
        # Transformer encoder
        self.transformer = TransformerEncoder(
            d_model=d_model,
            n_head=n_head,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout
        )
        
        # Classification head
        self.classifier = nn.Linear(d_model, 1)
        
    def forward(self, input_ids, attention_mask=None):
        batch_size = input_ids.size(0) // self.num_choices
        
        # Get embeddings
        embeddings = self.embeddings(input_ids)
        
        # Create attention mask for transformer (1 = attend, 0 = ignore)
        if attention_mask is not None:
            # Convert from (1 = keep, 0 = mask) to (False = keep, True = mask)
            transformer_attention_mask = attention_mask.eq(0)
        else:
            transformer_attention_mask = None
            
        # Pass through transformer
        sequence_outputs = self.transformer(embeddings, mask=transformer_attention_mask)
        
        # Use the [CLS] token (first token) for classification
        pooled_output = sequence_outputs[:, 0, :]
        
        # Get logits for each choice
        logits = self.classifier(pooled_output)
        
        # Reshape to [batch_size, num_choices]
        reshaped_logits = logits.view(batch_size, self.num_choices)
        
        return reshaped_logits